In [1]:
%env DATA_PATH=../../../data
from lib.fit import load_fit_file, get_gps_data, get_camera_starts, get_camera_ends, get_sensor_data
import json
import pandas as pd
import os
from tqdm.notebook import tqdm
import plotly.express as px
import subprocess
import numpy as np

DATA_PATH = '../../../data'

env: DATA_PATH=../../../data


In [2]:
fit = load_fit_file(f"{DATA_PATH}/archive/tmp/calibrate/2026-06-19-19-54-00.fit")

In [3]:
# s = stabalization, lc = lens correction
# idx_no_s = 2
# idx_no_s_lc = 3
# idx = idx_no_s_lc
idx = 0
camera_starts = get_camera_starts(fit)
camera_ends = get_camera_ends(fit)
start, end = camera_starts[idx], camera_ends[idx]
bag_folder = f"{DATA_PATH}/archive/tmp/calibrate/new_data{idx}"
os.makedirs(bag_folder, exist_ok=True)

## IMU

In [4]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }

In [5]:
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'alpha_x': 'accel_x', 'alpha_y': 'accel_y', 'alpha_z': 'accel_z'})
accel = accel_data.loc[start:end].drop(columns=['timestamp'])
accel *= -9.81 # expects specific force in m/s^2, and the accelerometer data is in g's
accel

,alpha_x,alpha_y,alpha_z
timestamp,,,
341522,0.215552,-0.934058,9.532178
341532,0.110171,-0.584385,9.378896
341542,-0.028740,-0.215552,9.551338
341552,-0.263452,-0.138911,9.838740
341562,-0.440684,-0.512534,9.944121
...,...,...,...
463574,-0.847837,1.365161,9.125024
463584,-0.459844,1.422642,9.374106
463594,-0.273032,1.096919,9.196875


In [6]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'omega_x': 'gyro_x', 'omega_y': 'gyro_y', 'omega_z': 'gyro_z'})
gyro = gyro_data.loc[start:end].drop(columns=['timestamp'])
gyro *= -(np.pi / 180) 
gyro

,omega_x,omega_y,omega_z
timestamp,,,
341522,-0.059597,-0.044697,0.063854
341532,-0.006385,-0.034055,0.022349
341542,0.038312,-0.037248,-0.075560
341552,0.058532,-0.026606,-0.117065
341562,0.042569,-0.020220,-0.063854
...,...,...,...
463574,-0.221359,-0.217102,0.219230
463584,-0.236258,-0.096844,0.121322
463594,-0.156441,0.102166,0.222423


In [7]:
px.line(accel)

In [8]:
gyro_mag = (gyro**2).sum(axis=1)**0.5
px.line(gyro_mag)

In [9]:
px.line(gyro)

In [11]:
imu = pd.merge_asof(gyro, accel, on='timestamp')
imu.timestamp = (imu.timestamp * 1e6).astype('int64')
imu.to_csv(f"{bag_folder}/imu0.csv", index=False, header=True)

In [87]:
with open(f"{DATA_PATH}/archive/tmp/calibrate/fit.json", 'w') as f:
    json.dump(fit, f, default=str, indent=2)

## Video

In [72]:
import cv2
# videos = ['VIRB0119.MP4', 'VIRB0120.MP4', 'VIRB0121.MP4', 'VIRB0122.MP4']
videos = ['VIRB0136.MP4']
p = f"{DATA_PATH}/archive/tmp/calibrate/{videos[idx]}"

In [45]:
cap = cv2.VideoCapture(p)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {p}")

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
progress = tqdm(total=frame_count, desc="Saving frames")

saved_frames = 0
os.makedirs(os.path.join(bag_folder, "cam0"), exist_ok=True)
STRIDE = 4
i = 0
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        i += 1
        if i % STRIDE != 0:
            continue
            
        timestamp_ms = start + cap.get(cv2.CAP_PROP_POS_MSEC)
        timestamp_ns = int(round(timestamp_ms * 1_000_000))
        out_path = os.path.join(bag_folder, f"cam0/{timestamp_ns}.png")
        
        cv2.imwrite(out_path, frame)
        saved_frames += 1
        progress.update(1)
finally:
    progress.close()
    cap.release()

saved_frames

Saving frames:   0%|          | 0/29336 [00:00<?, ?it/s]

7334

In [50]:
((end - start) / 1000) * 60

7558.5

## Camera Solve

In [88]:
# --- setup: intrinsics, aprilgrid detector, frame list, imu ---
import cv2, glob, yaml
from aprilgrid import Detector
from scipy.spatial.transform import Rotation
from scipy.signal import savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CALIB = f"{DATA_PATH}/archive/tmp/calibrate"
BAG = f"{CALIB}/fixed_data0"

# intrinsics (pinhole/radtan == opencv plumb_bob)
with open(f"{CALIB}/large-camchain.yaml") as f:
    cam = yaml.safe_load(f)['cam0']
fx, fy, cx, cy = cam['intrinsics']
K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
dist = np.array(cam['distortion_coeffs'])  # [k1, k2, p1, p2]
print('K =', K.ravel(), '\ndist =', dist)

# AprilGrid target. Use the Kalibr-native `aprilgrid` detector: its corner convention
# matches Kalibr's tagSize boundary, so the nominal yaml geometry is metrically correct
# as-is. NB: cv2.aruco's apriltag corners sit at a DIFFERENT boundary, so feeding the yaml
# geometry to an aruco object model gave ~10px reprojection and unusable poses.
with open(f"{CALIB}/april_6x6.yaml") as f:
    tgt = yaml.safe_load(f)
TAG_SIZE = tgt['tagSize']
PITCH = TAG_SIZE * (1 + tgt['tagSpacing'])   # center-to-center spacing
detector = Detector('t36h11')

def tag_object_points(tag_id):
    """Board-frame corners (order BL,BR,TR,TL, matching aprilgrid) for tag id = row*6+col."""
    gx, gy = tag_id % 6, tag_id // 6
    l, b = gx * PITCH, gy * PITCH
    return np.array([[l, b, 0], [l + TAG_SIZE, b, 0],
                     [l + TAG_SIZE, b + TAG_SIZE, 0], [l, b + TAG_SIZE, 0]], np.float32)

frames = sorted(glob.glob(f"{BAG}/cam0/*.png"),
                key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
imu_df = pd.read_csv(f"{BAG}/imu0.csv").sort_values('timestamp').reset_index(drop=True)
print(len(frames), 'frames,', len(imu_df), 'imu samples')

K = [886.88628831   0.         952.06904414   0.         885.08434813
 516.19584056   0.           0.           1.        ] 
dist = [-0.05240067  0.00170499 -0.0056583  -0.00147148]
7261 frames, 12080 imu samples


In [89]:
# --- per-frame camera pose from the aprilgrid (solvePnP) ---
STRIDE = 2
MIN_TAGS = 6        # planar PnP is ambiguous with few tags -> require a good spread
MAX_REPROJ = 3.0    # px; reject unstable solves

rows = []
prev_rv = prev_tv = None
for p in tqdm(frames[::STRIDE], desc='solve'):
    t_ns = int(os.path.splitext(os.path.basename(p))[0])
    dets = detector.detect(cv2.imread(p, cv2.IMREAD_GRAYSCALE))
    if len(dets) < MIN_TAGS:
        prev_rv = prev_tv = None
        continue
    obj = np.vstack([tag_object_points(d.tag_id) for d in dets]).astype(np.float32)
    img = np.vstack([np.array(d.corners).reshape(-1, 2) for d in dets]).astype(np.float32)
    # seed with the previous pose to avoid the planar 180-deg flip
    if prev_rv is not None:
        ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, rvec=prev_rv.copy(), tvec=prev_tv.copy(),
                                      useExtrinsicGuess=True)
    else:
        ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist)
    if not ok:
        prev_rv = prev_tv = None
        continue
    proj, _ = cv2.projectPoints(obj, rvec, tvec, K, dist)
    reproj = float(np.sqrt(((proj.reshape(-1, 2) - img) ** 2).sum(1).mean()))
    if reproj > MAX_REPROJ:
        prev_rv = prev_tv = None
        continue
    prev_rv, prev_tv = rvec, tvec
    R_cw, _ = cv2.Rodrigues(rvec)              # board -> camera
    pos = (-R_cw.T @ tvec).ravel()             # camera position in board frame
    q = Rotation.from_matrix(R_cw).as_quat()   # [x, y, z, w]
    rows.append((t_ns, *pos, q[3], q[0], q[1], q[2], len(dets), reproj, R_cw))

poses = pd.DataFrame(rows, columns=['timestamp', 'px', 'py', 'pz', 'qw', 'qx', 'qy', 'qz', 'n_tags', 'reproj', 'R_cw'])
print(f'{len(poses)} solved of {len(frames[::STRIDE])} ({len(poses)/max(1,len(frames[::STRIDE])):.0%}); '
      f'median reproj {poses.reproj.median():.2f}px')

# save in the same EuRoC layout as large-poses-cam0.csv
poses[['timestamp', 'px', 'py', 'pz', 'qw', 'qx', 'qy', 'qz']].to_csv(
    f"{BAG}/cam0-poses.csv", index=False,
    header=['#timestamp', ' p_RS_R_x [m]', ' p_RS_R_y [m]', ' p_RS_R_z [m]',
            ' q_RS_w []', ' q_RS_x []', ' q_RS_y []', ' q_RS_z []'])
poses.head()

solve:   0%|          | 0/3631 [00:00<?, ?it/s]

3380 solved of 3631 (93%); median reproj 1.38px


,timestamp,px,py,pz,qw,qx,qy,qz,n_tags,reproj,R_cw
0,25962000000,0.127698,0.192744,0.409258,-0.025128,0.999566,-0.014694,-0.004566,36,0.525579,"[[0.9995264708779829, -0.029605071892768245, -..."
1,25995366667,0.126775,0.192970,0.409273,-0.027332,0.999516,-0.013324,-0.006534,36,0.457154,"[[0.9995595871399536, -0.02699134396822608, -0..."
2,26028733333,0.126261,0.193139,0.409324,-0.027474,0.999524,-0.012021,-0.007275,36,0.510383,"[[0.9996051449834834, -0.02443002145970488, -0..."
3,26062100000,0.125328,0.193568,0.409173,-0.025999,0.999578,-0.010911,-0.006976,36,0.620483,"[[0.9996645836772792, -0.022175156388087317, -..."
4,26095466667,0.125456,0.193729,0.409521,-0.024653,0.999586,-0.012044,-0.008619,36,0.475526,"[[0.9995612879897575, -0.02450339462047672, -0..."


In [90]:
# --- camera angular velocity (camera frame) vs IMU gyro ---
P = poses.reset_index(drop=True)
w_rows = []
for i in range(len(P) - 1):
    dt = (P.timestamp[i + 1] - P.timestamp[i]) / 1e9
    if dt <= 0 or dt > 0.1:   # skip frame dropouts (>100ms gap)
        continue
    dR = P.R_cw[i] @ P.R_cw[i + 1].T          # camera-frame relative rotation
    w = Rotation.from_matrix(dR).as_rotvec() / dt
    w_rows.append(((P.timestamp[i] + P.timestamp[i + 1]) // 2, *w))
cam_w = pd.DataFrame(w_rows, columns=['timestamp', 'wx', 'wy', 'wz']).sort_values('timestamp').reset_index(drop=True)

if len(cam_w) > 11:  # light smoothing -- differentiation amplifies pose noise
    for c in ['wx', 'wy', 'wz']:
        cam_w[c] = savgol_filter(cam_w[c], 11, 2)

m = pd.merge_asof(cam_w, imu_df[['timestamp', 'omega_x', 'omega_y', 'omega_z']],
                  on='timestamp', direction='nearest')


fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=['x', 'y', 'z'])
for r, (cw, iw) in enumerate([('wx', 'omega_x'), ('wy', 'omega_y'), ('wz', 'omega_z')], start=1):
    fig.add_scatter(x=m.timestamp, y=m[cw], name=f'cam {cw}', legendgroup=cw, row=r, col=1)
    fig.add_scatter(x=m.timestamp, y=m[iw], name=f'imu {iw}', legendgroup=iw, row=r, col=1)
fig.update_layout(height=700, title='Angular velocity: camera (camera frame) vs IMU gyro')
fig

In [95]:
# compare imu gryo magnitude camera angular velocity magnitude
m['cam_w_mag'] = np.sqrt(m.wx**2 + m.wy**2 + m.wz**2)
m['imu_w_mag'] = np.sqrt(m.omega_x**2 + m.omega_y**2 + m.omega_z**2)
fig = px.line(m, x='timestamp', y=['cam_w_mag', 'imu_w_mag'], labels={'timestamp': 'timestamp [ns]', 'value': 'angular velocity magnitude [rad/s]'}, title='Angular velocity magnitude: camera vs IMU gyro')
fig.show()

In [96]:
from scipy.signal import correlate
# cross-correlate the magnitudes to find any time offset
corr = correlate(m.cam_w_mag - m.cam_w_mag.mean(), m.imu_w_mag - m.imu_w_mag.mean(), mode='full')
lags = np.arange(-len(m) + 1, len(m))
best_lag = lags[np.argmax(corr)]
print(f"Best lag: {best_lag} samples, {best_lag * (m.timestamp.diff().median() / 1e9):.3f} seconds")

Best lag: 2 samples, 0.067 seconds


In [94]:
# --- best-fit rotation mapping camera-frame omega -> imu-frame omega (Kabsch/SVD) ---
W = m[['wx', 'wy', 'wz']].to_numpy()
G = m[['omega_x', 'omega_y', 'omega_z']].to_numpy()
mask = np.linalg.norm(G, axis=1) > 0.15   # dynamic samples only; noise dominates when still
Wm, Gm = W[mask], G[mask]

H = Wm.T @ Gm
U, S, Vt = np.linalg.svd(H)
d = np.sign(np.linalg.det(Vt.T @ U.T))
R_cam_to_imu = Vt.T @ np.diag([1, 1, d]) @ U.T   # omega_imu ~= R_cam_to_imu @ omega_cam
resid = np.linalg.norm(Gm - (R_cam_to_imu @ Wm.T).T, axis=1)

print('R (cam -> imu) =\n', np.round(R_cam_to_imu, 3))
print('\nnearest signed permutation =\n', np.round(R_cam_to_imu).astype(int))
print(f'\nrms residual = {np.sqrt((resid ** 2).mean()):.4f} rad/s over {int(mask.sum())} samples')

R (cam -> imu) =
 [[ 0.999  0.004 -0.032]
 [ 0.032  0.017  0.999]
 [ 0.004 -1.     0.017]]

nearest signed permutation =
 [[ 1  0  0]
 [ 0  0  1]
 [ 0 -1  0]]

rms residual = 0.2042 rad/s over 2554 samples


In [92]:
# --- gravity-direction consistency (accel axis check) ---
# Gravity is fixed in the world, so g expressed in the board frame must be ~constant
# across frames at different camera orientations. Uses R_cam_to_imu from the cell above.
GYRO_STATIC = 0.05  # rad/s; loosen if too few static frames are found

acc = pd.merge_asof(P[['timestamp', 'R_cw']], imu_df, on='timestamp', direction='nearest')
gmag = np.linalg.norm(acc[['omega_x', 'omega_y', 'omega_z']].to_numpy(), axis=1)
static = acc[gmag < GYRO_STATIC].reset_index(drop=True)

gb = []
for _, r in static.iterrows():
    g_imu = np.array([r.alpha_x, r.alpha_y, r.alpha_z])
    g_cam = R_cam_to_imu.T @ g_imu   # imu -> camera
    gb.append(r.R_cw.T @ g_cam)      # camera -> board
gb = np.array(gb)

print(f'{len(gb)} static frames')
if len(gb):
    print('mean g_board =', np.round(gb.mean(0), 3), ' |g| =', round(float(np.linalg.norm(gb, axis=1).mean()), 3))
    print('std  g_board =', np.round(gb.std(0), 3), '  (small => accel & camera-rotation axes consistent)')
    fig2 = go.Figure()
    for i, c in enumerate('xyz'):
        fig2.add_scatter(y=gb[:, i], mode='markers', name=f'g_board {c}')
    fig2.update_layout(title='Gravity in board frame across static frames (should be flat)', height=400)
fig2 if len(gb) else 'no static frames -- raise GYRO_STATIC'

93 static frames
mean g_board = [ 0.208  9.648 -0.026]  |g| = 9.654
std  g_board = [0.157 0.128 0.189]   (small => accel & camera-rotation axes consistent)


## IMU Noise

In [16]:
fit = load_fit_file(f"{DATA_PATH}/archive/tmp/calibrate/2026-05-07-19-44-00.fit")

Caching archive_tmp_calibrate_2026-05-07-19-44-00.json


In [17]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
start, end = 4_000_000, 15_000_000

In [18]:
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'alpha_x': 'accel_x', 'alpha_y': 'accel_y', 'alpha_z': 'accel_z'})
accel = accel_data.loc[start:end].drop(columns=['timestamp'])
accel *= -9.81 # expects specific force in m/s^2, and the accelerometer data is in g's
accel

,alpha_x,alpha_y,alpha_z
timestamp,,,
4000006,-0.785566,-9.177715,0.464634
4000016,-0.790356,-9.168135,0.469424
4000026,-0.799937,-9.172925,0.459844
4000036,-0.804727,-9.196875,0.450264
4000046,-0.799937,-9.211245,0.450264
...,...,...,...
14999957,-0.799937,-9.177715,0.455054
14999967,-0.809517,-9.177715,0.455054
14999977,-0.804727,-9.187295,0.455054


In [ ]:
px.line(accel[:100_000])

In [19]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'omega_x': 'gyro_x', 'omega_y': 'gyro_y', 'omega_z': 'gyro_z'})
gyro = gyro_data.loc[start:end].drop(columns=['timestamp'])
gyro *= (np.pi / 180) 
gyro

,omega_x,omega_y,omega_z
timestamp,,,
4000006,-0.002128,-0.017028,0.012771
4000016,-0.001064,-0.017028,0.012771
4000026,-0.001064,-0.015963,0.012771
4000036,-0.001064,-0.017028,0.013835
4000046,-0.002128,-0.015963,0.014899
...,...,...,...
14999957,0.000000,-0.014899,0.014899
14999967,0.000000,-0.015963,0.014899
14999977,0.001064,-0.017028,0.014899


In [ ]:
px.line(gyro[:10_000])

In [20]:
imu = pd.merge_asof(accel, gyro, on='timestamp')
imu.timestamp = (imu.timestamp * 1e6).astype('int64')
imu.to_csv(f"{DATA_PATH}/archive/tmp/calibrate/noise_bag/imu0.csv", index=False, header=True)

In [24]:
(end - start) / 1000

11000.0